# 🦀 Bittensor 14 天共学营：一册到底交互式实战手册（`bittensor` v11.x）

> **欢迎来到 Bittensor 14 天共学营！**
> 为了避免在十几个终端窗口和脚本文件之间来回切换，我们将 **14 天全部核心原理、链下沙盒推演、本地链实操命令、矿工热修复填空与生产级部署** 完整浓缩到了这一份交互式 Jupyter Notebook 中。

### 🧭 如何使用本手册（标准 `.venv` 虚拟环境）
1. **步骤 1（创建标准 `.venv`）**：在终端进入 `camp-kit` 目录执行：
   ```bash
   cd camp-kit
   python3 -m venv .venv
   source .venv/bin/activate
   pip install -r requirements.txt
   ```
2. **步骤 2（在 VS Code / Cursor 中打开文件夹并选择 `.venv`）**：
   - **必须用 `File -> Open Folder...` 打开 `camp-kit` 整个文件夹**（不要直接双击打开单个 `.ipynb` 文件，否则无工作区路径时 VS Code 不会扫描 `.venv`）；
   - 若弹窗询问 *"Do you trust the authors of the files in this folder?"*（或左下角状态栏显示 **Restricted Mode**），请点击 **Yes, I trust the authors / Trust**（未信任的工作区会禁用 Python 环境扩展，导致看不到 `.venv`）；
   - 打开本 Notebook，点击右上角 **Select Kernel -> Python Environments...**，选择 **`.venv (Python 3.x)`**。
3. **步骤 3**：顺着章节从上往下点击每个代码单元格左侧的 **▶ (Run)** 按钮运行。


In [1]:
# [环境初始化与依赖一键自检] 首次打开或重启内核后请先运行本单元格
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

# 1. 动态定位当前 Notebook 所在项目根目录（不依赖文件夹名称，重命名为任意目录名均可自动识别）
def _is_project_root(d: Path) -> bool:
    return (d / "arena" / "common" / "chain.py").is_file() and (d / "requirements.txt").is_file()

candidates = []
for nb_var in (globals().get("__vsc_ipynb_file__"), os.environ.get("JPY_SESSION_NAME")):
    if nb_var:
        candidates.append(Path(nb_var).expanduser().resolve().parent)
cwd = Path.cwd().resolve()
candidates.extend([cwd, *cwd.parents, *(p for p in cwd.iterdir() if p.is_dir())])
for candidate in candidates:
    if _is_project_root(candidate):
        os.chdir(candidate)
        break
assert _is_project_root(Path.cwd()), f"未找到项目根目录（需包含 arena/ 与 requirements.txt），当前目录: {Path.cwd()}"

# 2. 将当前内核对应的 bin 目录与项目 .venv/bin 置顶写入 PATH（确保 !python / !btcli 与当前内核一致）
extra_paths = [str(Path(sys.executable).parent)]
local_venv_bin = Path.cwd() / ".venv" / "bin"
if local_venv_bin.is_dir() and str(local_venv_bin) not in extra_paths:
    extra_paths.insert(0, str(local_venv_bin))
extra_paths += [str(Path.home() / ".local" / "bin"), "/opt/homebrew/bin", "/usr/local/bin"]
curr_paths = [p for p in os.environ.get("PATH", "").split(":") if p and p not in extra_paths]
os.environ["PATH"] = ":".join(dict.fromkeys(extra_paths + curr_paths))
os.environ["TERM"] = "dumb"  # 禁用 CLI 动态旋转进度条以保持 Notebook 输出整洁

# 3. 检查核心依赖包，若漏装则自动在当前环境中补齐
REQUIRED_PACKAGES = {
    "bittensor": "bittensor>=11.3.0",
    "rich": "rich>=13.0.0",
    "openpyxl": "openpyxl>=3.1.0",
    "ipykernel": "ipykernel>=6.29.0",
}
missing = [spec for mod, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"📦 正在为当前环境自动补齐缺失依赖: {missing} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import bittensor as bt

btcli_path = shutil.which("btcli")
assert btcli_path is not None, "未找到 btcli 可执行文件"
print("✅ 环境与依赖自检通过！")
print(f"   • 当前工作目录 : {Path.cwd()}")
print(f"   • Python 解释器: {sys.executable} (Python {sys.version.split()[0]})")
print(f"   • Bittensor SDK: v{bt.__version__}")
print(f"   • btcli 路径   : {btcli_path}")


✅ 环境与依赖一键配置完成！
   • 当前工作目录 : /workspace/camp-kit
   • Python 内核  : /workspace/camp-kit/.venv/bin/python (Python 3.12.13)
   • Bittensor SDK: v11.3.0
   • btcli 路径   : /workspace/camp-kit/.venv/bin/btcli


---
## Day 1：Bittensor 全景与角色分工

Bittensor 把主网拆成多个按编号（`netuid`）隔离的子网。主网只负责记账、跑 Yuma 共识和按比例发币，具体任务与评分规则由各子网自行定义。

### 四类核心角色分工

| 角色 | 核心职责 | 收益来源 / 成本 |
| :--- | :--- | :--- |
| **子网拥有者（Subnet Owner）** | 定义任务协议（发什么题、交什么格式）与评分标准 | 获得子网排放分成（默认 18%）；需支付创建子网费用并维护开源仓库 |
| **矿工（Miner）** | 租用算力或存储，接收验证者请求并在时限内返回结果 | 按 `incentive`（激励占比）分得子网排放（默认占 41%）；承担注册费与硬件月租 |
| **验证者（Validator）** | 向矿工发题、核验结果质量与速度，定期向链上提交权重向量 | 按 `dividends`（分红占比）获得收益（与质押者共享 41%）；需持有高额质押以取得 `validator_permit` |
| **用户 / 质押者（Staker）** | 调用子网服务，或把 TAO 质押给看好的子网与验证者 | 分享验证者分红，同时承担子网 Alpha 代币价格波动风险 |

### 从请求到奖励的 5 步流转

1. **注册领号**：矿工与验证者在子网支付注册费，获得子网唯一编号 `UID`，矿工把自己的 `IP:Port` 写进链上元图（Metagraph）。
2. **签名发题**：验证者从元图读取在线矿工地址，用热密钥签名后通过 HTTP 发送任务请求。
3. **限时作答**：矿工核对验证者签名与许可证后，执行模型推理或数据提取，在超时前返回结果。
4. **打分上链**：验证者按子网规则给每个矿工算出得分，更新移动平均分（EMA），将归一化后的权重提交到链上。
5. **共识发币**：每过一个 `tempo`（主网默认 360 块，约 72 分钟），链上 Yuma 共识汇总所有验证者权重，剔除异常打分后向矿工和验证者发放排放。

### TAO 与子网 Alpha 代币的区别

| 代币类型 | 符号示例 | 核心定位 |
| :--- | :--- | :--- |
| **主网原生代币** | `TAO` ($\tau$) | 全网通用结算资产，用于跨子网交易、支付主网手续费及注入各子网流动性池 |
| **子网专属代币** | `Alpha` ($\alpha / \beta$) | dTAO 升级后每个子网独立的质押与排放凭证；把 TAO 质押进某个子网即通过子网池换入该子网的 Alpha 代币 |

> **Day 1 验收清单**：
> 1. 画出子网拥有者、验证者、矿工、用户之间的关系图，写清每个角色做什么。
> 2. 画出“从验证者发出请求到矿工拿到奖励”的 5 步流程图。
> 3. 用一句话说明 TAO 和子网 Alpha 的区别。

---
## Day 2：钱包与命令行（`btcli 11.x`）

### 2.1 版本核验与默认配置

开发包必须只安装 `bittensor`（11.x），**不要安装旧的独立包 `bittensor-cli`**（停留在 9.x，参数写法已淘汰）。下面检查版本并把默认网络设为 `finney` 主网：

In [2]:
# [Day 2.1] 检查 btcli 版本（必须为 11.x）并设置默认钱包与网络
!btcli --version
!btcli config set wallet camp
!btcli config set network finney
!btcli config get

11.3.0


set
    key  wallet
  value  camp
   path  /home/camp/.bittensor/btcli.json


set
    key  network
  value  finney
   path  /home/camp/.bittensor/btcli.json


config (/home/camp/.bittensor/btcli.json)


  network  finney
   wallet  camp


### 2.2 冷热密钥分离与钱包创建

Bittensor 钱包采用双层密钥设计，保护资金安全：

| 密钥类型 | 权限范围 | 存放位置与安全要求 |
| :--- | :--- | :--- |
| **冷密钥（Coldkey）** | 掌管资产转账、质押、解质押、子网注册付费 | 必须加密保管在本机或离线设备，**严禁上传到矿工云服务器** |
| **热密钥（Hotkey）** | 仅用于节点请求签名、公布 `IP:Port`、验证者提交权重 | 明文放在矿工/验证者服务器上供程序自动调用；即便泄露也无法转走冷密钥里的资金 |

下面用 Python SDK 创建一个名为 `camp` 的教学钱包（仅用于教学演示，请勿转入真实主网资产）：

In [3]:
# [Day 2.2] 创建教学钱包 camp 并列出本机钱包
import bittensor as bt

wallet = bt.Wallet(name="camp", hotkey="default")
if not wallet.coldkey_file.exists_on_device():
    wallet.create_new_coldkey(use_password=False, overwrite=False, suppress=True)
if not wallet.hotkey_file.exists_on_device():
    wallet.create_new_hotkey(use_password=False, overwrite=False, suppress=True)

print(f"钱包名称       : {wallet.name}")
print(f"冷密钥公开地址 : {wallet.coldkeypub.ss58_address}")
print(f"热密钥公开地址 : {wallet.hotkey.ss58_address}")
!btcli wallet list

钱包名称       : camp
冷密钥公开地址 : 5CaghQ2KhYa5M9hjydVM18iAY6FTvzSWVyr9tyiCPaHah3eX
热密钥公开地址 : 5Dhn64EUL19RSaXKA1caXQfZNqk1t7d6jZbWnr35idQoNabk
Wallets

~/.bittensor/wallets
└── camp  (sr25519)  (1 hotkey)
    5CaghQ2KhYa5M9hjydVM18iAY6FTvzSWVyr9tyiCPaHah3eX
    └── default  (sr25519)
        5Dhn64EUL19RSaXKA1caXQfZNqk1t7d6jZbWnr35idQoNabk

1 coldkey  ·  1 hotkey


### 2.3 查询主网 64 号子网与旧版命令改写练习

Bittensor 11.x 简化了命令行参数。旧教程里常见的 9.x 参数对照如下：

| 用途 | 旧版 9.x 写法（已废弃） | 新版 11.x 写法 |
| :--- | :--- | :--- |
| 指定钱包名 | `--wallet.name camp` | `-w camp` 或 `--wallet camp` |
| 指定热密钥 | `--wallet.hotkey default` | `-H default` 或 `--wallet-hotkey default` |
| 指定网络 | `--subtensor.network test` | `-n test` 或 `--network test` |
| 跳过交互确认（仅写入命令支持） | `--no-prompt` | `-y` 或 `--yes`（注：`wallet overview` 等只读查询命令无需此参数） |

> **动手练习（`TODO 0`）**：在下方单元格中，先查看主网 64 号子网（Chutes）的 `tempo`、注册价格和席位数（注意 `subnets show 64` 接收位置参数 `64`），再把旧版 9.x 命令改写为 11.x 合法写法：

In [4]:
# [Day 2.3 动手练习] 查询 64 号子网详情，并把 9.x 旧命令改写为 11.x 写法
import shlex
import subprocess

!btcli -n finney subnets show 64

old_cmd = "btcli wallet overview --wallet.name camp --subtensor.network test --no-prompt"
# TODO 0：把 old_cmd 改写为 btcli 11.x 的短参数写法（只读查询无需 -y）
new_cmd = "btcli wallet overview -w camp -n test"

print(f"\n旧版 9.x : {old_cmd}")
print(f"新版 11.x: {new_cmd}")

# 自动校验改写后的命令参数是否符合 btcli 11.x 规范
parts = shlex.split(new_cmd)
assert parts[:3] == ["btcli", "wallet", "overview"], "命令前缀应为 btcli wallet overview"
assert "-w" in parts or "--wallet" in parts, "请使用 -w 或 --wallet 指定钱包名"
assert "-n" in parts or "--network" in parts, "请使用 -n 或 --network 指定网络"
assert "--wallet.name" not in new_cmd and "--subtensor.network" not in new_cmd and "-y" not in parts, "请移除废弃参数与多余的 -y"
print("✅ TODO 0 命令改写校验通过！")

subnet 64 (Chutes)

          name  Chutes
         owner  5FRYKhbmfXPDoHdUUDMx27E3HuMvAzwjzFMMq3rNurUhAyS9
  owner hotkey  5CS3g6nVJM6ouns8n9buN9CzFf2C1YDHVcVGRcxoirKs2xbV
    registered  block 4,531,295  ~1.9y ago
         tempo  360
          burn  τ0.000500000
       neurons  256/256

identity
       github  https://github.com/chutesai/chutes
          url  https://chutes.ai
      discord  https://discord.gg/chutes
      contact  support@chutes.ai
  description  Breakthrough Serverless Compute for AI, At Scale.
         logo  https://storage.googleapis.com/chutes-random/logo-chutes.png
   additional  Chutes is a subnet operated by Chutes Global Corp, an International Business Corporation registered in Nevis with registration number C 61974



旧版 9.x : btcli wallet overview --wallet.name camp --subtensor.network test --no-prompt
新版 11.x: btcli wallet overview -w camp -n test
✅ TODO 0 命令改写校验通过！


> **Day 2 验收清单**：
> 1. 安装 Talisman 建一个教学钱包，截图 `5` 开头的公开地址，并在 Taostats 打开 64 号子网页面。
> 2. 截图 `btcli --version`（显示 11.x）、`btcli wallet list` 和 `btcli config get`。
> 3. 截图 `btcli -n finney subnets show 64` 输出，指出 64 号子网的 `tempo`、注册价格和席位数，并给出旧命令改写结果。
> 4. 回答：`coldkey` 和 `hotkey` 分别管什么？为什么矿工服务器上只放 `hotkey`？

---
## Day 3：dTAO、注册费与末位淘汰

### 3.1 注册费销毁与末位淘汰机制

| 概念 | 链上规则 |
| :--- | :--- |
| **注册费（Burn）** | 矿工注册时支付的注册费直接**销毁（不退还）**；有人成功注册时价格按子网配置的调整倍数跳升，无人注册时按区块逐步衰减 |
| **触发淘汰条件** | 当子网席位已满（如 `256 / 256`）且有新节点付费注册时，触发末位淘汰 |
| **新手保护期（Immunity Period）** | 新节点注册后的前 N 个区块（如 5000 块 $\approx 16.7$ 小时）处于保护期；正常情况下只从已过保护期的节点中淘汰（仅当极端情况下全子网节点都在保护期内时例外） |
| **淘汰排序依据** | 链在可淘汰节点中，优先淘汰 `pruning_score`（修剪分，由历史激励表现决定）最低的那一个 |

下面用 `btcli` 查询主网 64 号子网当前的注册费与新手保护期：

In [5]:
# [Day 3.1] 查询主网 64 号子网的实时注册费与新手保护期
!btcli query burn --netuid 64 --network finney
!btcli query immunity-period --netuid 64 --network finney

τ0.000500000


5000


### 3.2 动手写代码：保护期换算与注册预算判断

主网每 12 秒出一个区块（每小时 300 个区块）。

> **动手练习（`TODO 1`）**：调用 `sub.subnets.burn` 与 `sub.hyperparameters.immunity_period`，将保护期区块数换算为小时，并判断 1.0 TAO 预算是否足够注册：

In [6]:
# [Day 3.2 动手练习] 查询子网注册参数并换算保护期时长
import bittensor as bt

sub = bt.Subtensor("finney")
netuid = 64

burn_cost = sub.subnets.burn(netuid)
immunity_blocks = sub.hyperparameters.immunity_period(netuid)

# TODO 1：按每块 12 秒（每小时 300 块）换算保护期小时数，并判断 1.0 Alpha/TAO 预算是否够付注册费
immunity_hours = (immunity_blocks * 12) / 3600
my_budget = 1.0
can_afford = my_budget >= float(burn_cost.tao)

assert abs(immunity_hours - immunity_blocks / 300.0) < 1e-6, "保护期小时数换算有误，应为 immunity_blocks * 12 / 3600"
assert isinstance(can_afford, bool), "can_afford 应为布尔值"

print(f"子网 {netuid} 当前注册费 : {burn_cost}")
print(f"新手保护期             : {immunity_blocks} 区块 (约 {immunity_hours:.2f} 小时)")
print(f"预算 1.0 是否够付注册费: {'够用' if can_afford else '不够'}（✅ TODO 1 校验通过）")

子网 64 当前注册费 : τ0.000500000
新手保护期             : 5000 区块 (约 16.67 小时)
预算 1.0 是否够付注册费: 够用（✅ TODO 1 校验通过）


> **Day 3 验收清单**：
> 1. 任选一个主网子网，截图 `btcli query burn`、`btcli query immunity-period` 和 `btcli subnets show` 的输出，并把保护期区块数换算成小时。
> 2. 回答：注册费退不退？什么情况下才会发生淘汰？淘汰时链按什么排序？

---
## Day 4：读懂元图（Metagraph）

### 4.1 元图核心字段与角色判别

元图（Metagraph）是子网所有已注册节点的链上实时名册。看懂下面 6 个字段即可区分矿工与验证者：

| 字段 | 含义 | 矿工典型特征 | 验证者典型特征 |
| :--- | :--- | :--- | :--- |
| `UID` | 子网内的唯一座位号 | `0 ~ N-1` | `0 ~ N-1` |
| `stake` | 节点持有的质押权重 | 通常较低 | 高（靠质押排名前列取得验证许可） |
| `validator_permit` | 是否持有验证者许可证 | `False` | `True`（仅质押排名前 K 名持有） |
| `incentive` | 矿工激励占比（本子网内之和约为 1） | `> 0`（靠响应任务得分获取） | 通常为 `0` |
| `dividends` | 验证者分红占比（本子网内之和约为 1） | `0` | `> 0`（靠提交权重与共识一致获取） |
| `emission` | **每个 `tempo`（主网默认 360 块 $\approx 72$ 分钟）**发放的代币数 | 按 `incentive` 分得 | 按 `dividends` 分得 |

下面拉取主网 64 号子网元图，查看激励排名前 5 的矿工与分红排名前 3 的验证者：

In [7]:
# [Day 4.1] 拉取主网 64 号子网元图，对比矿工与验证者字段
import bittensor as bt

sub = bt.Subtensor("finney")
mg = sub.subnets.metagraph(64)

print(f"子网 64 总席位数: {mg.num_uids} | 当前区块: {mg.block} | Tempo: {mg.tempo}")
print("-" * 78)
print(f"{'角色':<8} | {'UID':<5} | {'Permit':<6} | {'Incentive':<10} | {'Dividends':<10} | {'Emission/Tempo'}")
print("-" * 78)

miners = sorted([n for n in mg.neurons if not n.validator_permit], key=lambda x: float(x.incentive), reverse=True)
validators = sorted([n for n in mg.neurons if n.validator_permit], key=lambda x: float(x.dividends), reverse=True)

for n in miners[:5]:
    print(f"{'矿工':<8} | {n.uid:<5} | {str(n.validator_permit):<6} | {float(n.incentive):<10.5f} | {float(n.dividends):<10.5f} | {n.emission}")
print("-" * 78)
for n in validators[:3]:
    print(f"{'验证者':<8} | {n.uid:<5} | {str(n.validator_permit):<6} | {float(n.incentive):<10.5f} | {float(n.dividends):<10.5f} | {n.emission}")

子网 64 总席位数: 256 | 当前区块: 9467972 | Tempo: 360
------------------------------------------------------------------------------
角色       | UID   | Permit | Incentive  | Dividends  | Emission/Tempo
------------------------------------------------------------------------------
矿工       | 207   | False  | 0.18408    | 0.00000    | 27.172195997ش
矿工       | 8     | False  | 0.06226    | 0.00000    | 9.189910940ش
矿工       | 82    | False  | 0.05759    | 0.00000    | 8.500691292ش
矿工       | 52    | False  | 0.05655    | 0.00000    | 8.347302386ش
矿工       | 14    | False  | 0.05248    | 0.00000    | 7.747347242ش
------------------------------------------------------------------------------
验证者      | 1     | True   | 0.00000    | 0.38431    | 56.725953782ش
验证者      | 252   | True   | 0.00000    | 0.25370    | 37.447834284ش
验证者      | 59    | True   | 0.00000    | 0.21755    | 32.111683795ش


### 4.2 动手写代码：估算榜首矿工 24 小时产出

主网默认 `tempo = 360` 区块（$360 \times 12\text{ 秒} = 72\text{ 分钟} = 1.2\text{ 小时}$），因此 **24 小时共有 $24 / 1.2 = 20$ 个 `tempo`**。

> **动手练习（`TODO 2`）**：筛选出无验证者许可证的纯矿工节点，并按每天 20 个 `tempo` 估算榜首矿工的日产出：

In [8]:
# [Day 4.2 动手练习] 筛选纯矿工并计算榜首矿工的日产出
import bittensor as bt

if "mg" not in globals():
    mg = bt.Subtensor("finney").subnets.metagraph(64)

# TODO 2：筛选出 validator_permit 为 False 的矿工节点，并按 incentive 从高到低排序
pure_miners = [n for n in mg.neurons if not n.validator_permit]
pure_miners.sort(key=lambda n: float(n.incentive), reverse=True)

assert len(pure_miners) > 0, "未找到 validator_permit=False 的节点"
top_miner = pure_miners[0]
tempos_per_day = (24 * 3600) / (mg.tempo * 12)  # 20.0 个 tempo/天
daily_alpha_output = float(top_miner.emission.alpha) * tempos_per_day
assert abs(tempos_per_day - 86400 / (mg.tempo * 12)) < 1e-6, "每天 tempo 数量计算有误"

print(f"无验证许可节点数量 : {len(pure_miners)} / {mg.num_uids}")
print(f"榜首矿工节点 UID   : {top_miner.uid} (Incentive: {float(top_miner.incentive):.5f})")
print(f"单 Tempo 周期排放  : {top_miner.emission}")
print(f"预估 24h 日产出    : {daily_alpha_output:.2f} Alpha/天（✅ TODO 2 校验通过）")

无验证许可节点数量 : 239 / 256
榜首矿工节点 UID   : 207 (Incentive: 0.18408)
单 Tempo 周期排放  : 27.172195997ش
预估 24h 日产出    : 543.44 Alpha/天（✅ TODO 2 校验通过）


> **Day 4 验收清单**：
> 1. 同一个子网，分别截图 Taostats 页面和 `btcli query metagraph --netuid <编号>`（或上方元图表格）的输出，标出同一个节点的对应行。
> 2. 圈出一个矿工和一个验证者，说明你是根据哪些字段判断的（`incentive`、`dividends`、`validator_permit`）。
> 3. 写出 `emission` 字段的单位和统计周期，并估算该矿工一天的产出。

---
## Day 5：Yuma 共识与 Commit-Reveal

### 5.1 Yuma 共识如何惩罚偏袒小号的验证者

当某个验证者给自己的作弊小号矿工打满分 `1.00`、而其他诚实验证者都打 `0.00` 时，Yuma 共识的单周期核心裁剪流程按三步结算：

| 步骤 | 计算逻辑 | 作用 |
| :--- | :--- | :--- |
| **1. 质押加权中位数** | 对每名矿工，将验证者打分从高到低排序并累加验证者质押，累计质押跨过 $\kappa = 0.50$（50%）那一档的打分即为共识上限 $\bar{W}_j$ | 找到过半数质押认可的最高合理分 |
| **2. 异常高分削顶（Clipping）** | 将超出共识上限的打分直接削平：$\tilde{W}_{ij} = \min(W_{ij}, \bar{W}_j)$ | 让偏袒小号的虚高打分无法生效 |
| **3. 验证者对齐度与简化分红** | 模拟器按保留下来的有效权重占比计算验证者信任度 $\text{vTrust}_i = \sum_j \tilde{W}_{ij} / \sum_j W_{ij}$，再乘以质押计算单周期分红 | 作弊验证者的权重被削光，`vTrust` 与分红随之归零 |

> **模型边界说明**：教学模拟器 `tools/day05_yuma_sim.py` 用单周期模型展示最核心的 **50% 质押截断与 `vTrust` 惩罚**，省略了真实链上跨周期的 **历史平滑债券（EMA Bonds）与 Liquid Alpha** 机制。

脚本中设定了 **3 个验证者（`Validator_A` 占 20% 质押、`Validator_B` 占 45% 质押、`Validator_C` 占 35% 质押）** 与 **5 个矿工（`Miner_1`～`Miner_5`）**。下面运行模拟器，观察 `Validator_A` 给作弊小号 `Miner_5` 打 `1.00` 满分时的裁剪结果：

In [9]:
# [Day 5.1] 运行 Yuma 共识模拟器（Validator_A 占 20% 质押，给作弊小号 Miner_5 打 1.00 分）
!python tools/day05_yuma_sim.py

╭──────────────────────────────────────────────────────────────────────────────╮
│ Bittensor Day 5：Yuma 共识（Yuma Consensus）异常权重削弱演示                 │
│ 场景：3 个验证者给 5 个矿工打分。其中 Validator_A (占 20% 质押) 试图作弊，   │
│ 把所有权重 1.00 全打给了自己的作弊小号 Miner_5，而另外两个诚实验证者给       │
│ Miner_5 打了 0 分。                                                          │
╰──────────────────────────────────────────────────────────────────────────────╯
        📊 第一部分：矿工权重裁剪对比（看 Miner_5 的作弊高分如何被削掉）        
┏━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┓
┃          ┃    Val_A ┃    Val_B ┃     Val_C ┃     若无 ┃      Yuma ┃     Yuma ┃
┃          ┃     打分 ┃     打分 ┃      打分 ┃     Yuma ┃  共识上限 ┃ 裁剪后 … ┃
┃          ┃   (Stake ┃   (Stake ┃    (Stake ┃     保护 ┃ (Consens… ┃   (Final ┃
┃ 矿工节点 ┃     20%) ┃     45%) ┃      35%) ┃ (简单加… ┃     阈值) ┃ Incenti… ┃
┡━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━┩
│ Miner_1  │     0.00 │     0.40 │      0.35 │    30.2% │      0.35 │

�─────────────┼──────────────────┼───────────────────┼───────────────────┤
│ Validator_B       │              45% │             95.0% │             56.2% │
│ (正常评测节点 1)  │                  │                   │                   │
├───────────────────┼──────────────────┼───────────────────┼───────────────────┤
│ Validator_C       │              35% │             95.0% │             43.7% │
│ (正常评测节点 2)  │                  │                   │                   │
└───────────────────┴──────────────────┴───────────────────┴───────────────────┘



✅ 已同步生成可交互 Excel 模拟表：/workspace/camp-kit/tools/day05_yuma_simulator.xlsx


### 5.2 动手实验：测试 $\kappa = 0.50$ 质押加权中位数边界

在上面的默认场景中，对 `Miner_5` 打分从高到低排序为：`Validator_A (1.00, 质押 20%) -> Validator_B (0.00, 质押 45%) -> Validator_C (0.00, 质押 35%)`。累加质押在 `Validator_B` 处达到 `20% + 45% = 65% >= 50%`，因此 `Miner_5` 的共识上限是 **`0.00`**。

下面我们直接调用 `day05_yuma_sim.py` 中的 `compute_weighted_consensus` 与 `run_yuma_simulation`，对比验证两个关键边界场景：
1. **场景 1（诚实验证者 B 给了 `Miner_5` 微弱分 `0.05`）**：此时累加质押在 `Validator_B`（`20% + 45% = 65%`）处跨过 `50%`，`Miner_5` 的共识上限从 `0.00` 变为 **`0.05`**，但 `Validator_A` 打的 `1.00` 依然被削掉了 **95%**（削平到 `0.05`）！
2. **场景 2（若作弊验证者 A 自身掌握了 `55%` 过半质押）**：一旦作弊方质押单独超过 `50%`，共识上限就会被拉到 `1.00`——这也直观解释了为什么 Yuma 共识的前提是“诚实质押占多数（$>50\%$）”。

In [10]:
# [Day 5.2 动手实验] 直接调用 compute_weighted_consensus 验证 κ=0.50 阈值边界
from tools.day05_yuma_sim import compute_weighted_consensus

stakes_default = [0.20, 0.45, 0.35]  # Validator A (20%), B (45%), C (35%)

# 默认异常场景：A 打 1.00，B 打 0.00，C 打 0.00
th_default = compute_weighted_consensus(stakes_default, [1.00, 0.00, 0.00], kappa=0.50)

# 场景 1：A 打 1.00，B 给了一点微弱分 0.05，C 打 0.00 -> 累计质押 0.20 + 0.45 = 0.65 >= 0.50，落在 B 的 0.05
th_scene1 = compute_weighted_consensus(stakes_default, [1.00, 0.05, 0.00], kappa=0.50)
clipped_a_scene1 = min(1.00, th_scene1)

# 场景 2：若作弊方 A 掌握 55% 质押 ([0.55, 0.25, 0.20]) -> 第一档就跨过 0.50
th_scene2 = compute_weighted_consensus([0.55, 0.25, 0.20], [1.00, 0.00, 0.00], kappa=0.50)

assert th_default == 0.00, "默认场景下 Miner_5 共识上限应为 0.00"
assert th_scene1 == 0.05 and clipped_a_scene1 == 0.05, "场景 1 共识上限应为 0.05"
assert th_scene2 == 1.00, "场景 2 过半质押作弊时上限应为 1.00"

print("场景对比                          | 质押分布 (A/B/C)   | Miner_5 原始打分   | κ=0.50 共识上限 | Val_A 裁剪后有效分")
print("-" * 102)
print(f"默认场景（A 作弊，B/C 打 0）      | [0.20, 0.45, 0.35] | [1.00, 0.00, 0.00] |      {th_default:.2f}       |       {min(1.0, th_default):.2f}")
print(f"场景 1（B 给 0.05 微弱分）        | [0.20, 0.45, 0.35] | [1.00, 0.05, 0.00] |      {th_scene1:.2f}       |       {clipped_a_scene1:.2f} (削去 95%)")
print(f"场景 2（若作弊方 A 握有 55% 质押）| [0.55, 0.25, 0.20] | [1.00, 0.00, 0.00] |      {th_scene2:.2f}       |       {min(1.0, th_scene2):.2f} (过半失守)")
print("\n✅ Yuma 共识 κ=0.50 质押加权中位数边界实验核验通过！")

场景对比                          | 质押分布 (A/B/C)   | Miner_5 原始打分   | κ=0.50 共识上限 | Val_A 裁剪后有效分
------------------------------------------------------------------------------------------------------
默认场景（A 作弊，B/C 打 0）      | [0.20, 0.45, 0.35] | [1.00, 0.00, 0.00] |      0.00       |       0.00
场景 1（B 给 0.05 微弱分）        | [0.20, 0.45, 0.35] | [1.00, 0.05, 0.00] |      0.05       |       0.05 (削去 95%)
场景 2（若作弊方 A 握有 55% 质押）| [0.55, 0.25, 0.20] | [1.00, 0.00, 0.00] |      1.00       |       1.00 (过半失守)

✅ Yuma 共识 κ=0.50 质押加权中位数边界实验核验通过！


### 5.3 Commit-Reveal（加密提交与延迟揭示）

如果验证者明文提交权重，懒惰验证者可以直接抄袭大验证者的权重来白嫖分红。开启 `Commit-Reveal` 后，权重提交时处于时间锁加密状态，过一个揭示周期后才在链上解密公开。

- **为什么能防抄袭**：别人看到明文权重时已经是几个周期前的历史分数，无法实时跟风抄作业。
- **什么情况下防不住**：如果子网里的矿工排名长期一成不变，抄袭者即使照抄上一周期的旧权重，依然能蹭到相近的得分。

下面查询主网 64 号子网是否开启了 `commit-reveal`：

In [11]:
# [Day 5.3] 查询主网 64 号子网是否启用 Commit-Reveal
!btcli query commit-reveal-enabled --netuid 64 --network finney

False


> **Day 5 验收清单**：
> 1. 截图 `python tools/day05_yuma_sim.py` 输出，说明异常打分如何被削减、作弊验证者分红如何下降，并写出模拟器省略了真实机制中的哪一项（如 EMA 债券累积 Bonds）。
> 2. 截图 `btcli query commit-reveal-enabled --netuid 64` 的输出。
> 3. 回答：`commit-reveal` 为什么能限制抄权重？它在什么情况下防不住？

---
## Day 6：子网选型与成本测算

下场当矿工前，必须先根据各子网官方仓库说明算清楚硬件月租与沉没注册费。下面对比课程第 06 课拆解的三类典型子网：

| 对比维度 | 推理类子网（SN64 Chutes / SN1 Apex） | 数据类子网（SN13 Data Universe） |
| :--- | :--- | :--- |
| **矿工交付什么** | 托管开源大模型，响应流式推理或特定提示词生成任务 | 持续抓取、清洗并存储指定来源（如 X/Reddit）的公开语料数据集 |
| **验证者按什么打分** | 1. 答案质量/对数概率校验（防小模型伪造）<br>2. 首字延迟（TTFT）与吞吐速度（Tokens/s） | 1. 数据真实性与去重校验（抽查原帖与时间戳是否一致）<br>2. 数据新鲜度、覆盖主题广度与总容量 |
| **硬件要求（以各子网仓库为准）** | 高端 GPU（如 SN64 常见多卡 A100 / H100 80GB 节点） | 多核 CPU + 大容量 NVMe SSD + 住宅代理 IP 池（无需高端 GPU） |
| **按 Runpod / 云主机参考估算月成本** | 单卡/多卡 GPU 实例常见数百至数千美元/月（随卡型与时长浮动） | 大容量存储型 VPS + 代理流量常见约 \$80 ~ \$200 / 月（随抓取规模浮动） |

下面用 Python SDK 实时拉取主网 `SN1`、`SN13`、`SN64` 的当前注册费、在网节点席位数与保护期：

In [12]:
# [Day 6] 实时查询主网 SN1 (Apex)、SN13 (Data Universe)、SN64 (Chutes) 的注册费、席位规模与保护期
import bittensor as bt

sub = bt.Subtensor("finney")
print(f"{'子网':<18} | {'当前注册费 (Burn)':<16} | {'席位 (在网/上限)':<16} | {'保护期 (块/小时)':<18} | {'Tempo'}")
print("-" * 86)

for netuid, name in [(1, "SN1  Apex"), (13, "SN13 DataUniverse"), (64, "SN64 Chutes")]:
    burn = sub.subnets.burn(netuid)
    hp = sub.subnets.subnet_hyperparameters(netuid)
    mg_info = sub.subnets.metagraph(netuid)
    imm_blocks = int(hp.get("immunity_period", 0))
    max_n = int(hp.get("max_validators", 0)) or mg_info.num_uids
    seats_str = f"{mg_info.num_uids} 节点"
    tempo = int(hp.get("tempo", 360))
    hours = (imm_blocks * 12) / 3600
    print(f"{name:<18} | {str(burn):<16} | {seats_str:<16} | {imm_blocks:>5} 块 ({hours:>5.1f}h) | {tempo} 块")

子网                 | 当前注册费 (Burn)     | 席位 (在网/上限)       | 保护期 (块/小时)         | Tempo
--------------------------------------------------------------------------------------


SN1  Apex          | τ0.000500000     | 256 节点           |  7200 块 ( 24.0h) | 99 块


SN13 DataUniverse  | τ0.036700610     | 256 节点           | 12000 块 ( 40.0h) | 360 块


SN64 Chutes        | τ0.000500000     | 256 节点           |  5000 块 ( 16.7h) | 360 块


### 预算 200 美元/月的选型思路

- **成本约束**：200 美元/月难以负担 7×24 小时在线的多卡 A100/H100 推理服务器，但可以覆盖入门级数据子网的存储型 VPS 与基础代理支出（具体回本周期需结合当期注册销毁费与链上排放单独核算，切勿将未变现的代币估值直接冲抵现金账单）。
- **技术门槛与稳定性**：数据类子网核心拼的是抓取管道稳定性、去重与索引响应速度，适合通过确定性工程优化逐步提分；而在重算力推理子网，算力不足会因超时得 0 分并在保护期结束后被淘汰。

> **Day 6 验收清单**：
> 1. 完成推理类子网（SN64 或 SN1）与数据类子网（SN13）的五项对比表（交付物、打分标准、硬件、当前注册费、按云主机公开报价估算的月成本）。
> 2. 回答：预算每月 200 美元，你会先选哪一个？从成本、技术门槛、得分稳定性三方面说明。
> 3. 选做：注册 Chutes 账户，获取 API Key。

---
## Day 7：第一周复盘与第二周环境预检

第二周将在本机启动一条完整的 Docker 本地教学链（`subtensor-localnet`）。在进入 Day 8 前，运行 `tools/day07_check_env.py` 一次性检查：
1. Python 运行环境（`>= 3.10`）；
2. `bittensor` 11.x SDK 与 `btcli` 版本一致性；
3. Docker 服务状态与本地教学链镜像（`ghcr.io/raofoundation/subtensor-localnet:devnet`）；
4. 第二周实战代码包（`arena/`、`scripts/`、`tools/`）文件完整性；
5. 本地钱包目录与提取引擎配置。

In [13]:
# [Day 7] 运行第二周本地链环境预检脚本（首次拉取镜像可加 --pull-image）
!python tools/day07_check_env.py

🔍 Bittensor 共学营 Day 7 实操环境自检表
1. Python 运行环境 (>=3.10)      | Python 3.12.13                               | ✅ PASS
2. Bittensor 11.x 工具链        | SDK=11.3.0 | 11.3.0                          | ✅ PASS
3. Docker 与本地链镜像             | Docker 正常，教学链镜像已就绪                           | ✅ PASS
4. 第二周实战代码包              | 本地链/矿工/陪练/验证者/监控脚本齐全                         | ✅ PASS
5. 钱包与提取引擎配置                 | 钱包=camp | 引擎=内置规则 | ✅ PASS
🎉 恭喜！你的第 2 周环境自检全部通过，可直接截图提交 Day 7 作业！


> **Day 7 验收清单**：
> 1. 补交 Day 1–6 未完成的题目。
> 2. 启动 Docker，运行 `python tools/day07_check_env.py --pull-image`，截图所有检查项通过。
> 3. 在群里提一个具体问题（例如某个子网的评分漏洞或回本周期计算）。

---
## Day 8：启动本地教学链并注册身份

### 8.1 启动本地链与一键初始化教学子网

第二周使用两个脚本初始化本地链环境：

| 脚本 | 关键动作 | 重要技术细节 |
| :--- | :--- | :--- |
| `scripts/start_localnet.sh` | 启动 `local_chain` 容器（监听 `ws://127.0.0.1:9944`） | 传入 `False` 启用 **12 秒标准出块模式**；若用快速出块模式，注册交易的防抢跑加密时期（Era）会过期报错。注意容器使用 `--rm` 模式，若停掉容器需重新运行 `setup_localnet.sh` |
| `scripts/setup_localnet.sh` | 创建教学子网（默认 `netuid 2`），注册验证者、你的矿工与两个陪练矿工，并给验证者质押 `30 TAO` | 将 `max_allowed_validators` 设为 `2`（仅子网拥有者 `UID 0` 与验证者 `UID 1` 持有 `validator_permit`）、`tempo` 设为 `10` 块（约 2 分钟）、`weights_set_rate_limit` 设为 `5` 块 |

下面检查本地教学链容器、RPC 连通性以及四个角色的链上注册状态（若容器未启动或子网未初始化会自动执行初始化）：

In [14]:
# [Day 8.1] 检查本地教学链容器与子网注册状态（未初始化则自动执行初始化）
import subprocess
import sys
from pathlib import Path
import bittensor as bt
from arena.common.chain import default_netuid, load_wallet

# 1. 检查 local_chain 容器是否在线
res = subprocess.run(
    ["docker", "ps", "--filter", "name=^/local_chain$", "--format", "{{.Names}} ({{.Status}})"],
    capture_output=True, text=True, check=True,
)
if "local_chain" not in res.stdout:
    print("⏳ 正在启动本地教学链容器 (start_localnet.sh)...")
    subprocess.run(["bash", "scripts/start_localnet.sh"], check=True)
else:
    print(f"✅ 本地链容器已在线: {res.stdout.strip()}")

# 2. 检查链上教学子网与 4 个节点身份是否已完整注册（防止容器重启后链状态清空但旧文件残留）
sub_local = bt.Subtensor("ws://127.0.0.1:9944")
netuid = default_netuid()
ROLES = [("validator", "default"), ("miner", "default"), ("miner", "bot_fast"), ("miner", "bot_slow")]

def check_all_registered(n_uid: int) -> bool:
    try:
        for w_name, hk in ROLES:
            w = load_wallet(w_name, hk)
            if not w.hotkey_file.exists_on_device():
                return False
            if sub_local.neurons.uid(w.hotkey.ss58_address, n_uid) is None:
                return False
        return True
    except Exception:
        return False

if not check_all_registered(netuid):
    print("⏳ 检测到链上教学子网或节点尚未初始化，正在运行 setup_localnet.sh...")
    subprocess.run(["bash", "scripts/setup_localnet.sh"], check=True)
    netuid = default_netuid()

print(f"\n📋 本地教学链（netuid={netuid}）节点注册核验表：")
print("-" * 76)
for w_name, hk in ROLES:
    w = load_wallet(w_name, hk)
    uid = sub_local.neurons.uid(w.hotkey.ss58_address, netuid)
    assert uid is not None, f"{w_name}/{hk} 未在子网 {netuid} 注册成功"
    print(f"钱包 {w_name + '/' + hk:<18} | 热密钥 {w.hotkey.ss58_address[:14]}... | 子网 {netuid} UID: {uid}")

✅ 本地链容器已在线: local_chain (Up 12 minutes (healthy))

📋 本地教学链（netuid=2）节点注册核验表：
----------------------------------------------------------------------------
钱包 validator/default  | 热密钥 5G8vaQHxfucVDJ... | 子网 2 UID: 1
钱包 miner/default      | 热密钥 5Dt3HchWKr6XNT... | 子网 2 UID: 2
钱包 miner/bot_fast     | 热密钥 5DhuoLXeZZD7ua... | 子网 2 UID: 3
钱包 miner/bot_slow     | 热密钥 5CWx7dRYrM9XrJ... | 子网 2 UID: 4


### 8.2 检查本地链元图与验证许可（`validator_permit`）

下面读取本地链 `netuid 2` 的元图，核对哪个 `UID` 拥有 `validator_permit`，以及为什么矿工没有：

In [15]:
# [Day 8.2] 查看本地链教学子网元图，核对 validator_permit 与质押分布
import bittensor as bt
from arena.common.chain import default_netuid

sub_local = bt.Subtensor("ws://127.0.0.1:9944")
netuid = default_netuid()
mg_local = sub_local.subnets.metagraph(netuid)
print(f"本地链子网 {netuid} 节点数: {mg_local.num_uids} | 当前区块: #{mg_local.block} | Tempo: {mg_local.tempo}")
print("-" * 78)
print(f"{'UID':<5} | {'Total Stake':<18} | {'Validator Permit':<18} | {'Incentive':<10} | {'IP:Port'}")
print("-" * 78)
for n in mg_local.neurons:
    print(f"{n.uid:<5} | {str(n.total_stake):<18} | {str(n.validator_permit):<18} | {float(n.incentive):<10.4f} | {n.axon or '未发布'}")

本地链子网 2 节点数: 5 | 当前区块: #62 | Tempo: 10
------------------------------------------------------------------------------
UID   | Total Stake        | Validator Permit   | Incentive  | IP:Port
------------------------------------------------------------------------------
0     | 23.168713234β      | True               | 0.0000     | 未发布
1     | 39.073771783β      | True               | 0.0000     | 未发布
2     | 0.000000000β       | False              | 0.0000     | 未发布
3     | 0.000000000β       | False              | 0.0000     | 未发布
4     | 0.000000000β       | False              | 0.0000     | 未发布


### 8.3 动手写代码：校验热密钥注册状态

矿工与验证者启动的第一步，都是在代码里自动检查自己的热密钥是否已在目标子网注册。

> **动手练习（`TODO 3`）**：调用 `sub_local.neurons.netuids_for_hotkey` 与 `sub_local.neurons.uid`，查询 `miner/default` 的注册状态与 `UID`：

In [16]:
# [Day 8.3 动手练习] 用 SDK 检查 miner/default 是否已在教学子网注册并获取 UID
import bittensor as bt
from arena.common.chain import default_netuid, load_wallet

sub_local = bt.Subtensor("ws://127.0.0.1:9944")
netuid = default_netuid()
mg_local = sub_local.subnets.metagraph(netuid)
miner_wallet = load_wallet("miner", "default")
hk_address = miner_wallet.hotkey.ss58_address

# TODO 3：检查 hk_address 是否在目标 netuid 上注册，并读取对应的 UID
registered_netuids = sub_local.neurons.netuids_for_hotkey(hk_address)
is_reg = netuid in registered_netuids
my_uid = sub_local.neurons.uid(hk_address, netuid)

assert is_reg and my_uid is not None, f"矿工热密钥 {hk_address} 尚未在子网 {netuid} 注册"
print(f"矿工热密钥地址 : {hk_address}")
print(f"已注册子网列表 : {registered_netuids} (是否在子网 {netuid} 注册: {is_reg})")
print(f"分配的链上 UID : {my_uid} (验证许可 validator_permit = {mg_local.neurons[my_uid].validator_permit}，✅ TODO 3 校验通过)")

矿工热密钥地址 : 5Dt3HchWKr6XNTf9yndSy6dAJ5Fp6nU6oNhCavdsycwMTwuQ
已注册子网列表 : [2] (是否在子网 2 注册: True)
分配的链上 UID : 2 (验证许可 validator_permit = False，✅ TODO 3 校验通过)


> **Day 8 验收清单**：
> 1. 运行 `start_localnet.sh` 和 `setup_localnet.sh`，截图注册结果并标出你的矿工 `UID`。
> 2. 查看本地链元图，指出哪个 `UID` 有 `validator_permit`？为什么矿工没有？（提示：`setup_localnet.sh` 将子网 `max_allowed_validators` 设为 2，只有质押排名前 2 的子网拥有者 `UID 0` 与验证者 `UID 1` 拿到许可证）。
> 3. 回答：注册时链上提示的价格，下一个人紧接着注册时会怎么变化？如果很久没人注册又会怎么变化？

---
## Day 9：跑起矿工，完成首次交付

### 9.1 教学子网任务协议（`arena/common/protocol.py`）

第二周的教学子网模拟「中文快讯结构化提取」任务。验证者通过 `POST /extract` 向矿工发题，协议与三关评分规则如下：

| 方向 / 阶段 | 字段 / 关卡 | 规则说明 |
| :--- | :--- | :--- |
| **验证者请求** | `raw_text`、`required_fields`、`timeout` | 传入中文快讯原文，要求在 `3.0` 秒内提取 `["项目名称", "融资金额", "领投机构", "子网编号"]` 四个字段 |
| **矿工响应** | `extracted_data`、`evidence_quotes` | 返回四个字段值，并为每个非空字段附上一句**在原文中一字不差的原话**作为依据 |
| **空值约定** | `""`（空字符串） | 原文未提及某字段（如“未披露投资方名单”）时，提取值与原文依据都必须填 `""` |
| **第一关（门禁）** | 超时、缺字段、依据不在原文中 | 任一不满足直接判 **`0.0000` 分**（防止大模型瞎编幻觉） |
| **第二关（70%）** | 字段准确率 | 四个字段与参考答案一致的比例，占总分 `70%` |
| **第三关（30%）** | 响应速度 | 耗时越短得分越高，占总分 `30%` |

### 9.2 端点发布防坑：为什么不能写 `127.0.0.1` 或代理虚拟 IP？

矿工启动时调用 `client.execute(bt.ServeAxon(...), wallet)` 把自己的 `IP:Port` 写进链上元图，有两个常见网络坑：
- **为什么链拒绝 `127.0.0.1`**：链上安全规则禁止发布回环地址（否则所有外部验证者都会连到自己机器上）。
- **为什么开代理软件会导致收不到请求**：开启 Clash / Surge 的 TUN 模式时，系统会多出一张 `198.18.0.1` 虚拟网卡；如果把虚拟网卡 IP 写上链，其他节点根本连不通。本项目的 `arena/common/chain.py`（`lan_ip()`）已自动过滤 `198.18.x.x` 并优先选取真实局域网 IP（如 `192.168.x.x`）。

### 9.3 启动矿工与陪练，完成首次评分（观察初始 Bug 导致的 0 分）

现在确保你的初始矿工（`camp_miner.py`，端口 `8091`）和两个陪练矿工（`practice_bots.py`，端口 `8092`、`8093`）在线，然后直接运行验证者只评分模式（`--score-only`），看看未经修改的初始矿工会得多少分：

In [17]:
# [Day 9.3] 启动初始矿工（含两处待修缺陷，传 --serve 发布链上端点）与陪练矿工，观察首次 3 轮评分现场
import re
import socket
import subprocess
import sys
import time
from pathlib import Path
from arena.common.chain import default_netuid, ensure_port_free, fetch_metagraph

# 0. 若之前运行过 Day 12 (PM2) 或 Day 13 (Docker)，先清理守护进程以免抢占 8091-8093 端口
subprocess.run("pm2 delete camp-miner camp-bots camp-monitor >/dev/null 2>&1 || true", shell=True)
subprocess.run("docker compose -f arena/day13_docker/docker-compose.yml down >/dev/null 2>&1 || true", shell=True)

# 1. 确保 camp_miner.py 处于 Day 9 的初始待排错状态（保留完整的“从错误状态 -> 排错 -> 正确状态”教学过程）
miner_file = Path("arena/day09_10_miner/camp_miner.py")
BUGGY_BLOCK = '''def extract_field(field: str, text: str) -> str:
    m = None
    for p in PATTERNS.get(field, []):
        m = re.search(p, text)
        if m:
            break
    return m.group(1).strip()


def make_evidence(field: str, value: str, text: str) -> str:
    return f"{field}：{value}"'''

FIXED_BLOCK = '''def extract_field(field: str, text: str) -> str:
    for p in PATTERNS.get(field, []):
        m = re.search(p, text)
        if m:
            return m.group(1).strip()
    return ""


def make_evidence(field: str, value: str, text: str) -> str:
    for s in re.split(r"[。！？]", text):
        if value and value in s:
            return s.strip()
    return ""'''

src_text = miner_file.read_text(encoding="utf-8")
if FIXED_BLOCK in src_text:
    miner_file.write_text(src_text.replace(FIXED_BLOCK, BUGGY_BLOCK), encoding="utf-8")
assert BUGGY_BLOCK in miner_file.read_text(encoding="utf-8"), "camp_miner.py 初始代码块校验失败"

def wait_port(port: int, timeout: float = 25.0) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("127.0.0.1", port)) == 0:
                return True
        time.sleep(0.4)
    return False

# 2. 启动两个陪练矿工（bot_fast: 8092, bot_slow: 8093）与你的初始矿工（8091，加 --serve 发布链上端点）
subprocess.run([sys.executable, "arena/day09_10_miner/practice_bots.py", "-d"], check=True)
assert wait_port(8092, 25.0) and wait_port(8093, 25.0), "陪练矿工 (8092/8093) 启动超时"

subprocess.run([sys.executable, "arena/day09_10_miner/camp_miner.py", "--serve", "-d"], check=True)
assert wait_port(8091, 25.0), "学员矿工 (8091) 启动超时"

netuid = default_netuid()
mg_now = fetch_metagraph("ws://127.0.0.1:9944", netuid)
published = [f"UID {n.uid}@{n.axon}" for n in mg_now.neurons if n.axon]
print(f"✅ 链上已登记端点的节点: {', '.join(published)}\n")

# 3. 清空旧 EMA 状态，运行验证者 3 轮首次评分（--score-only --seed 12 --rounds 3）
Path(".localnet/validator_scores.json").unlink(missing_ok=True)
subprocess.run([sys.executable, "arena/day11_validator/camp_validator.py", "--score-only", "--seed", "12", "--rounds", "3"], check=True)

print("\n📄 你的矿工实时报错日志 (.localnet/miner.log 末尾关键行)：")
print("-" * 76)
for line in Path(".localnet/miner.log").read_text(encoding="utf-8").splitlines()[-12:]:
    print("  ", line)

✅ 已把 192.168.50.194:8092 写上链（UID 3），区块 0x07236ccfeef16ba7cba58f5101da8bb24581a7760c58674b543b043a71346f27
✅ 已把 192.168.50.194:8093 写上链（UID 4），区块 0xf5030e0b92890b0d4d464dfddd596a9aacdbb1f8fc899184edbd2e38e51096ee
🤖 两个陪练矿工已在后台启动（PID 11046，监听 192.168.50.194:8092 / 8093）
📄 实时日志：/workspace/camp-kit/.localnet/bots.log（可执行 tail -f .localnet/bots.log 查看）


✅ 已把 192.168.50.194:8091 写上链（UID 2），区块 0xec57d40f528e9e2936a87af305add8c828aacf1b6cee6758a40c28da4937e9ef
🚀 矿工已在后台启动（PID 11761，监听 192.168.50.194:8091）
📄 实时日志：/workspace/camp-kit/.localnet/miner.log（可执行 tail -f .localnet/miner.log 查看）
✅ 链上已登记端点的节点: UID 2@192.168.50.194:8091, UID 3@192.168.50.194:8092, UID 4@192.168.50.194:8093



🔑 第 1 步  验证者 UID 1  质押 44.221489077β  验证许可=True  区块 #71
📡 第 2 步  找到 3 个已发布端点的矿工：UID 2@192.168.50.194:8091，UID 3@192.168.50.194:8092，UID 4@192.168.50.194:8093

📝 第 1 轮  task_7252：【教学快讯】Bittensor 生态项目 Beacon 宣布完成 6000万美元 战略轮融资。本轮融资由 Blue Harbor Fund 领投。项目位于 33号 子网，团队表示资金将用于扩充算力节点。
   UID 2   耗时  0.02s  本轮 0.0000  EMA 0.0000→0.0000  依据不在原文中：[项目名称]『项目名称：Beacon』
   UID 3   耗时  0.02s  本轮 0.0000  EMA 0.0000→0.0000  依据不在原文中：[项目名称]『AI 总结：数据真实可靠』
   UID 4   耗时  2.02s  本轮 0.6229  EMA 0.0000→0.1869  通过（字段正确 3/4）

📝 第 2 轮  task_4729：【教学简讯】Delta 团队透露已完成 4200万���元 融资，但未披露投资方名单。该项目目前运行在 33号 子网。
   UID 2   耗时  0.00s  本轮 0.0000  EMA 0.0000→0.0000  没有拿到结果（HTTP 500）
   UID 3   耗时  0.00s  本轮 0.0000  EMA 0.0000→0.0000  依据不在原文中：[项目名称]『AI 总结：数据真实可靠』
   UID 4   耗时  2.01s  本轮 0.7988  EMA 0.1869→0.3705  通过（字段正确 4/4）

📝 第 3 轮  task_3658：【教学周报】算力网络 Cobalt（58号 子网）公布季度数据。本季度完成 1500万美元 新一轮融资，领投方为 Lakeside Ventures，新资金主要用于数据中心建设。
   UID 2   耗时  0.00s  本轮 0.0000  EMA 0.0000→0.0000  依据不在原文中：[项目名称]『项目名称：Cobalt』
   UID 3 

可以看到首次运行的完整故障现场：
1. **第 1 轮（`task_7252`）与第 3 轮（`task_3319`）**：你的矿工（`UID 2`）成功收到了验证者 `UID 1` 的签名请求并提取出了字段，但得分为 **`0.0000`**，扣分原因是 **`依据不在原文中`**（返回了拼接的 `『项目名称：Beacon』`）。
2. **第 2 轮（`task_4729`，原文写“未披露投资方名单”）**：你的矿工在处理缺失字段时直接抛出 `AttributeError: 'NoneType' object has no attribute 'group'`，导致検証者收到 **`HTTP 500`**。
3. **最终权重**：慢速陪练 `UID 4` 凭着 `0.4464` 的平均分独占了 **`1.000`（100%）** 权重。下面我们在 Day 10 逐一诊断并修复这两处缺陷！

> **Day 9 验收清单**：
> 1. 启动矿工与陪练，截图元图中三个矿工 `UID` 均已登记非零的 `ip:port`。
> 2. 运行 `python arena/day11_validator/camp_validator.py --score-only --seed 12`，截图矿工收到验证者签名请求的日志，以及验证者输出的各矿工得分与扣分原因。
> 3. 回答：为什么链拒绝 `127.0.0.1`？开代理时为什么自动探测的 IP（`198.18.x.x`）不能用？矿工靠哪两步确认请求来自合法的验证者？

---
## Day 10：矿工排错与代码修复

### 10.1 两道安全门禁与 `403 Forbidden` 拦截复现

矿工在执行提取任务前，必须用两道门禁拦截伪造请求与矿工互刷：

| 门禁 | 源码调用 | 拦截结果 |
| :--- | :--- | :--- |
| **1. 签名核验** | `bt.http_auth.verify` | 签名与发送方热密钥不匹配时，返回 `HTTP 401 Unauthorized` |
| **2. 许可证核验** | `mg.neurons[caller_uid].validator_permit` | 调用方在链上无验证者许可证时，返回 `HTTP 403 Forbidden` |

运行 `diagnose_node.py --test-403`，一次性检查本机端口监听、链上端点一致性、`ulimit -n`，并用无验证许可的 `miner` 热密钥向自己发起请求，验证 `HTTP 403 Forbidden` 拦截：

In [18]:
# [Day 10.1] 运行矿工自检与无证调用 403 拦截测试
!python arena/day09_10_miner/diagnose_node.py --test-403

🩺 第 10 天矿工节点诊断报告（网络=local，netuid=2，区块=#71）
1. 本机端口监听       | 127.0.0.1:8091                 | OPEN（监听中）              | ✅ 正常
2. 链上登记端点       | 链上=192.168.50.194:8091      | 本机=192.168.50.194:8091 | ✅ 一致（UID 2）
3. 文件描述符上限     | ulimit -n                     | Soft=1048575           | ✅ 够用
4. 无验证许可拦截测试 | caller=miner/default（permit=False） | HTTP 403 Forbidden（✅ 成功拦截无验证许可的 hotkey：{"error": "caller has no validator permit on this subnet"}）


### 10.2 定位并修复 `camp_miner.py` 的两处代码缺陷

打开 `arena/day09_10_miner/camp_miner.py` 查看第 47～78 行，找到导致 Day 9 得 `0` 分或报 `HTTP 500` 的两处缺陷：

In [19]:
# [Day 10.2] 查看 camp_miner.py 第 47~78 行的两处预留缺陷
from pathlib import Path

miner_file = Path("arena/day09_10_miner/camp_miner.py")
lines = miner_file.read_text(encoding="utf-8").splitlines()
for idx in range(46, min(78, len(lines))):
    print(f"{idx+1:3d}: {lines[idx]}")

 47:     "融资金额": [r"(\d+万美元)"],
 48:     "领投机构": [r"由 (.+?) 领投", r"领投方为 (.+?)，"],
 49:     "子网编号": [r"(\d+号) ?子网"],
 50: }
 51: 
 52: 
 53: # ---------------------------------------------------------------- 提取逻辑
 54: def extract_field(field: str, text: str) -> str:
 55:     m = None
 56:     for p in PATTERNS.get(field, []):
 57:         m = re.search(p, text)
 58:         if m:
 59:             break
 60:     return m.group(1).strip()
 61: 
 62: 
 63: def make_evidence(field: str, value: str, text: str) -> str:
 64:     return f"{field}：{value}"
 65: 
 66: 
 67: def extract_local(text: str, fields: list) -> Tuple[Dict[str, str], Dict[str, str], str]:
 68:     data, quotes = {}, {}
 69:     for f in fields:
 70:         v = extract_field(f, text)
 71:         data[f] = v
 72:         quotes[f] = make_evidence(f, v, text) if v else ""
 73:     return data, quotes, "local-rules"
 74: 
 75: 
 76: def extract_chutes(text: str, fields: list) -> Tuple[Dict[str, str], Dict[str, str], str]:
 7

| 缺陷位置 | 错误原因 | 修复方案 |
| :--- | :--- | :--- |
| **缺陷 1：`make_evidence()`** | 返回自己拼接的 `f"{field}：{value}"`，这句话不在 `text` 原文中，触发第一关一票否决（`0` 分） | 用 `re.split(r"[。！？]", text)` 按句切分原文，返回包含 `value` 的那句原话 `s.strip()` |
| **缺陷 2：`extract_field()`** | 当所有正则都未匹配（如第 2 轮原文写“未披露投资方名单”）时，`m` 为 `None`，调用 `m.group(1)` 直接抛 `AttributeError` 导致 `HTTP 500` | 匹配成功时立即 `return m.group(1).strip()`，循环结束未匹配则返回空字符串 `""` |

下面将这两处修复写入 `arena/day09_10_miner/camp_miner.py` 并保存（供后续 Day 11 上链、Day 12 PM2 守护和 Day 13 Docker 容器构建持续使用），重启 `8091` 矿工，并在完全相同的 3 道题目（`--seed 12 --rounds 3`）上复测：

In [20]:
# [Day 10.3] 修复 camp_miner.py 两处缺陷 -> 重启矿工 -> 在相同 3 道题 (--seed 12) 上复测逆袭夺冠
import socket
import subprocess
import sys
import time
from pathlib import Path

miner_file = Path("arena/day09_10_miner/camp_miner.py")
current_src = miner_file.read_text(encoding="utf-8")

old_block = '''def extract_field(field: str, text: str) -> str:
    m = None
    for p in PATTERNS.get(field, []):
        m = re.search(p, text)
        if m:
            break
    return m.group(1).strip()


def make_evidence(field: str, value: str, text: str) -> str:
    return f"{field}：{value}"'''

fixed_block = '''def extract_field(field: str, text: str) -> str:
    for p in PATTERNS.get(field, []):
        m = re.search(p, text)
        if m:
            return m.group(1).strip()
    return ""


def make_evidence(field: str, value: str, text: str) -> str:
    for s in re.split(r"[。！？]", text):
        if value and value in s:
            return s.strip()
    return ""'''

if old_block in current_src:
    miner_file.write_text(current_src.replace(old_block, fixed_block), encoding="utf-8")
    print("✅ 已将 extract_field() 与 make_evidence() 的修复写入 camp_miner.py！")
elif fixed_block in current_src:
    print("ℹ️ camp_miner.py 已处于修复后状态。")
else:
    raise RuntimeError("未找到预期代码块，请检查 camp_miner.py")

# 重启修复后的 8091 矿工服务（后台模式 -d 会自动安全释放旧 8091 进程并启动新进程）
_ = subprocess.run([sys.executable, "arena/day09_10_miner/camp_miner.py", "-d"], check=True)

port_ready = False
for _ in range(30):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(("127.0.0.1", 8091)) == 0:
            port_ready = True
            break
    time.sleep(0.3)
assert port_ready, "修复后的矿工 (8091) 启动失败，请检查 .localnet/miner.log"

# 在完全相同的 3 道题目 (--seed 12 --rounds 3) 上一次性连续复测
Path(".localnet/validator_scores.json").unlink(missing_ok=True)
_ = subprocess.run([sys.executable, "arena/day11_validator/camp_validator.py", "--score-only", "--seed", "12", "--rounds", "3"], check=True)

✅ 已将 extract_field() 与 make_evidence() 的修复写入 camp_miner.py！


ℹ️ 端口 8091 被上次残留的后台进程（PID 11761）占用，已自动停止旧进程并释放端口。
🚀 矿工已在后台启动（PID 11856，监听 192.168.50.194:8091）
📄 实时日志：/workspace/camp-kit/.localnet/miner.log（可执行 tail -f .localnet/miner.log 查看）


🔑 第 1 步  验证者 UID 1  质押 44.221489077β  验证许可=True  区块 #71
📡 第 2 步  找到 3 个已发布端点的矿工：UID 2@192.168.50.194:8091，UID 3@192.168.50.194:8092，UID 4@192.168.50.194:8093

📝 第 1 轮  task_7252：【教学快讯】Bittensor 生态项目 Beacon 宣布完成 6000万美元 战略轮融资。本轮融资由 Blue Harbor Fund 领投。项目位于 33号 子网，团队表示资金将用于扩充算力节点。
   UID 2   耗时  0.02s  本轮 0.9978  EMA 0.0000→0.2993  通过（字段正确 4/4）
   UID 3   耗时  0.00s  本轮 0.0000  EMA 0.0000→0.0000  依据不在原文中：[项目名称]『AI 总结：数据真实可靠』
   UID 4   耗时  2.01s  本轮 0.6239  EMA 0.0000→0.1872  通过（字段正确 3/4）

📝 第 2 轮  task_4729：【教学简讯】Delta 团队透露已完成 4200万美元 融资，但未披露投资方���单。该项目目前运行在 33号 子网。
   UID 2   耗时  0.00s  本轮 0.9997  EMA 0.2993→0.5094  通过（字段正确 4/4）
   UID 3   耗时  0.00s  本轮 0.0000  EMA 0.0000→0.0000  依据不在原文中：[项目名称]『AI 总结：数据真实可靠』
   UID 4   耗时  2.01s  本轮 0.7993  EMA 0.1872→0.3708  通过（字段正确 4/4）

📝 第 3 轮  task_3658：【教学周报】算力网络 Cobalt（58号 子网）公布季度数据。本季度完成 1500万美元 新一轮融资，领投方为 Lakeside Ventures，新资金主要用于数据中心建设。
   UID 2   耗时  0.00s  本轮 0.9999  EMA 0.5094→0.6565  通过（字段正确 4/4）
   UID 3   耗时  0.00s  本轮 0.0000  EMA 0.0000

修复后，你的矿工（`UID 2`）在三道不同题型（含第 2 轮缺失字段题 `task_4729`）上全部通过三关校验，单轮得分均达到 **`0.997+ ~ 0.999+`**，三轮累积移动平均分（EMA）升至 **`0.656+`**，折算归一化权重占比达 **`0.595`（59.5%）**，反超 4 号陪练（`0.405`）夺回全子网第一名！

> **Day 10 验收清单**：
> 1. 截图 `python arena/day09_10_miner/diagnose_node.py --test-403` 的三项检查结果与第 4 项 `HTTP 403 Forbidden` 拦截结果。
> 2. 截图你修改 `camp_miner.py` 的代码行，以及修复后在 `--seed 12 --rounds 3` 上连续 3 轮得分高于陪练矿工的输出。
> 3. 回答：真实子网里，矿工在线却一直收不到验证者请求，按顺序列出你会检查的 3 个地方（1. 链上元图 `ip:port` 是否非零且与宿主机对外地址一致；2. 云主机防火墙/安全组与代理虚拟网卡 `198.18.x.x`；3. 调用方是否通过签名与 `validator_permit` 检查）。

---
## Day 11：成为验证者，提交权重

### 11.1 动手写代码：EMA 移动平均与权重归一化

在本项目的验证者脚本（`camp_validator.py`）中，为了避免矿工单轮偶发抖动直接决定上链权重，验证者在本地对每轮得分维护指数移动平均分（EMA，$\alpha = 0.3$），再将各矿工的 EMA 归一化成总和为 `1.0` 的权重向量提交上链（注：默认每次运行从 0 开始计算 3 轮 EMA；加 `--keep-ema` 参数则会跨次加载 `.localnet/validator_scores.json` 持续累积）：

$$\text{EMA}_i^{(t)} = \alpha \cdot s_i^{(t)} + (1 - \alpha) \cdot \text{EMA}_i^{(t-1)}, \quad w_i = \frac{\text{EMA}_i^{(t)}}{\sum_j \text{EMA}_j^{(t)}}$$

> **动手练习（`TODO 4`）**：按平滑系数 $\alpha = 0.3$，根据上一轮历史分 `prev_ema` 和本轮新得分 `current_scores` 计算更新后的 `new_ema` 与归一化权重 `weights`（需兼容总分为 0 的边界情况）：

In [21]:
# [Day 11.1 动手练习] 实现验证者的 EMA 移动平均更新与权重归一化
alpha = 0.3
prev_ema = {2: 0.8000, 3: 0.0000, 4: 0.6500}
current_scores = {2: 0.9999, 3: 0.0000, 4: 0.6800}

# TODO 4：计算更新后的 EMA 字典 new_ema，以及归一化后的权重字典 weights（总分为 0 时返回全 0）
new_ema = {uid: alpha * current_scores[uid] + (1 - alpha) * prev_ema[uid] for uid in current_scores}
total_ema = sum(new_ema.values())
weights = {uid: (val / total_ema if total_ema > 0 else 0.0) for uid, val in new_ema.items()}

assert abs(new_ema[2] - 0.85997) < 1e-5 and abs(sum(weights.values()) - 1.0) < 1e-6, "EMA 或权重归一化计算有误"

print("UID | 上轮 EMA | 本轮得分 | 更新后 EMA | 归一化上链权重")
print("-" * 58)
for uid in sorted(weights):
    print(f" {uid}  |  {prev_ema[uid]:.4f}  |  {current_scores[uid]:.4f}  |   {new_ema[uid]:.4f}   |    {weights[uid]:.4f}")
print(f"权重总和校验: {sum(weights.values()):.4f}（✅ TODO 4 校验通过）")

UID | 上轮 EMA | 本轮得分 | 更新后 EMA | 归一化上链权重
----------------------------------------------------------
 2  |  0.8000  |  0.9999  |   0.8600   |    0.5662
 3  |  0.0000  |  0.0000  |   0.0000   |    0.0000
 4  |  0.6500  |  0.6800  |   0.6590   |    0.4338
权重总和校验: 1.0000（✅ TODO 4 校验通过）


### 11.2 正式运行验证者，将加密权重提交上链

去掉 `--score-only` 参数运行 `camp_validator.py`。验证者完成发题打分后，会调用 `bt.SetWeights` 挤入區塊并触发本地链的 Timelock `Commit-Reveal` 加密提交：

In [22]:
# [Day 11.2] 运行完整验证者流程：发题 -> 打分 -> 更新 EMA -> 提交权重上链 (bt.SetWeights)
import subprocess
import sys
import time
import bittensor as bt
from arena.common.chain import default_netuid, load_wallet

# 若距离上一次提交权重不足 weights_rate_limit (5 个区块)，自动等待几个区块再提交
sub_local = bt.Subtensor("ws://127.0.0.1:9944")
netuid = default_netuid()
val_w = load_wallet("validator", "default")
val_uid = sub_local.neurons.uid(val_w.hotkey.ss58_address, netuid)

rc = subprocess.run([sys.executable, "arena/day11_validator/camp_validator.py", "--seed", "12", "--rounds", "3"]).returncode
if rc != 0:
    print("⏳ 距离上次提交不足 5 个区块 (weights_rate_limit)，等待 1 个区块周期后自动重试...")
    time.sleep(13)
    _ = subprocess.run([sys.executable, "arena/day11_validator/camp_validator.py", "--seed", "12", "--rounds", "3"], check=True)

print("\n🔍 提交后立即读取链上权重（观察 Commit-Reveal 加密期现象）：")
_ = subprocess.run([sys.executable, "arena/day11_validator/camp_validator.py", "--show-weights"], check=True)

🔑 第 1 步  验证者 UID 1  质押 44.221489077β  验证许可=True  区块 #72
📡 第 2 步  找到 3 个已发布端点的矿工：UID 2@192.168.50.194:8091，UID 3@192.168.50.194:8092，UID 4@192.168.50.194:8093

📝 第 1 轮  task_7252：【教学快讯】Bittensor 生态项目 Beacon 宣布完成 6000万美元 战略轮融资。本轮融资由 Blue Harbor Fund 领投。项目位于 33号 子网，团队表示资金将用于扩充算力节点。
   UID 2   耗时  0.01s  本轮 0.9994  EMA 0.0000→0.2998  通过（字段正确 4/4）
   UID 3   耗时  0.00s  本轮 0.0000  EMA 0.0000→0.0000  依据不在原文中：[项目名称]『AI 总结：数据真实可靠』
   UID 4   耗时  2.01s  本轮 0.6242  EMA 0.0000→0.1873  通过（字段正确 3/4）

📝 第 2 轮  task_4729：【教学简讯】Delta 团队透露已完成 4200万美元 融资，但未披露投资方���单。该项目目前运行在 33号 子网。
   UID 2   耗时  0.00s  本轮 0.9998  EMA 0.2998→0.5098  通过（字段正确 4/4）
   UID 3   耗时  0.00s  本轮 0.0000  EMA 0.0000→0.0000  依据不在原文中：[项目名称]『AI 总结：数据真实可靠』
   UID 4   耗时  2.01s  本轮 0.7990  EMA 0.1873→0.3708  通过（字段正确 4/4）

📝 第 3 轮  task_3658：【教学周报】算力网络 Cobalt（58号 子网）公布季度数据。本季度完成 1500万美元 新一轮融资，领投方为 Lakeside Ventures，新资金主要用于数据中心建设。
   UID 2   耗时  0.00s  本轮 0.9999  EMA 0.5098→0.6568  通过（字段正确 4/4）
   UID 3   耗时  0.00s  本轮 0.0000  EMA 0.0000

🔑 第 1 步  验证者 UID 1  质押 44.221489077β  验证许可=True  区块 #75

⛓️  链上当前生效的权重（验证者 UID 1）： （暂无，可能还在 reveal 周期内）
   全部验证者的权重行：{}


### 11.3 读取链上解密生效的权重与矿工盯盘

本地教学链开启了自动 `Commit-Reveal`，刚提交时权重处于时间锁加密状态。等待约 2 分钟（跨过揭示周期）后，运行 `--show-weights` 查看链上已解密生效的权重，并用 `uid_monitor.py` 连续 3 轮监测矿工 `UID 2` 的排名、`incentive` 与新增排放：

In [23]:
# [Day 11.3] 等待本地链跨越 Reveal 解密与 Tempo 结算块后，查看链上生效权重并监测矿工收益
import subprocess
import sys
import time
import bittensor as bt
from arena.common.chain import default_netuid, load_wallet

sub_local = bt.Subtensor("ws://127.0.0.1:9944")
netuid = default_netuid()
val_uid = sub_local.neurons.uid(load_wallet("validator", "default").hotkey.ss58_address, netuid)
miner_uid = sub_local.neurons.uid(load_wallet("miner", "default").hotkey.ss58_address, netuid)

print(f"⏳ 正在等待子网 {netuid} 完成 Commit-Reveal 自动解密与 Tempo 周期结算...")
revealed_ok = False
for i in range(30):
    rows = sub_local.weights.weights(netuid)
    my_weights = rows.get(val_uid) or {}
    mg_check = sub_local.subnets.metagraph(netuid)
    miner_inc = float(mg_check.neurons[miner_uid].incentive)
    if my_weights and miner_inc > 0.5:
        revealed_ok = True
        print(f"✅ 区块 #{mg_check.block}：权重已解密生效，且矿工 UID {miner_uid} 激励已更新 (Incentive={miner_inc:.4f})！")
        break
    time.sleep(6)

assert revealed_ok, "等待权重解密或 Tempo 结算超时，请检查 local_chain 容器状态"
_ = subprocess.run([sys.executable, "arena/day11_validator/camp_validator.py", "--show-weights"], check=True)
_ = subprocess.run([sys.executable, "arena/day11_validator/uid_monitor.py", "--uid", str(miner_uid), "--rounds", "3", "--interval", "12"], check=True)

⏳ 正在等待子网 2 完成 Commit-Reveal 自动解密与 Tempo 周期结算...


✅ 区块 #87：权重已解密生效，且矿工 UID 2 激励已更新 (Incentive=0.5952)！


🔑 第 1 步  验证者 UID 1  质押 53.340971543β  验证许可=True  区块 #87

⛓️  链上当前生效的权重（验证者 UID 1）： {2: 0.5951991717072639, 4: 0.4048008282927361}
   全部验证者的权重行：{1: {2: 0.5951991717072639, 4: 0.4048008282927361}}


📡 监控 UID 2  网络 local  netuid 2  保护期 5000 块
[1] 区块 #87  UID 2  排名 1/5  incentive 0.5952  trust 0.0000  pruning_score 0.0000  上个 epoch 排放 2.440330223β  tempo 10（约每天 720 个 epoch）  保护期 剩 4922 块  axon 192.168.50.194:8091  ✅ 正常


[2] 区块 #88  UID 2  排名 1/5  incentive 0.5952  trust 0.0000  pruning_score 0.0000  上个 epoch 排放 2.440330223β  tempo 10（约每天 720 个 epoch）  保护期 剩 4921 块  axon 192.168.50.194:8091  ✅ 正常


[3] 区块 #89  UID 2  排名 1/5  incentive 0.5952  trust 0.0000  pruning_score 0.0000  上个 epoch 排放 2.440330223β  tempo 10（约每天 720 个 epoch）  保护期 剩 4920 块  axon 192.168.50.194:8091  ✅ 正常


> **Day 11 验收清单**：
> 1. 截图 `python arena/day11_validator/camp_validator.py` 的验证者 `UID`/质押/许可证、各矿工本轮得分/EMA/归一化权重，以及链返回的提交成功区块哈希。
> 2. 等待揭示周期后，截图 `python arena/day11_validator/camp_validator.py --show-weights` 读回的链上生效权重。
> 3. 截图 `python arena/day11_validator/uid_monitor.py --rounds 3 --interval 12` 连续 3 次读取矿工排名、`incentive`、排放和保护期状态。
> 4. 回答：为什么验证者用移动平均分（EMA）而不是只看最新一轮？开启 `commit-reveal` 的子网上，提交权重后立刻读回为什么还看不到新权重？

---
## Day 12：用 PM2 守护进程

矿工节点需要 7×24 小时在线响应验证者抽查。直接在前台终端跑 `python camp_miner.py`，一旦 SSH 断开或进程异常退出就会掉线失分。生产环境常用 PM2 守护矿工与监控服务（本项目 `arena/day12_pm2/ecosystem.config.js` 同时守护 `camp-miner`、`camp-bots` 和 `camp-monitor` 三个进程）：

| 操作目标 | 标准命令 |
| :--- | :--- |
| **启动全部守护进程** | `pm2 start arena/day12_pm2/ecosystem.config.js` |
| **查看在线状态与 PID** | `pm2 list` |
| **模拟崩溃自愈（验证重启次数 `↺` 加 1）** | `kill -9 $(pm2 pid camp-miner)` 后再执行 `pm2 list` |
| **安装并配置 50MB 日志轮转** | `pm2 install pm2-logrotate && pm2 set pm2-logrotate:max_size 50M` |

下面查看 `arena/day12_pm2/ecosystem.config.js` 配置，并在本机实测 PM2 启动、模拟进程崩溃自动拉起，以及重启后验证者发题打分：

In [24]:
# [Day 12] 查看 PM2 配置，实测 PM2 守护启动、kill -9 崩溃自愈与重启后评分验证
import shutil
import subprocess
import sys
import time
from pathlib import Path

print("=== arena/day12_pm2/ecosystem.config.js ===")
print(Path("arena/day12_pm2/ecosystem.config.js").read_text(encoding="utf-8"))

if shutil.which("pm2"):
    _ = subprocess.run("pm2 delete camp-miner camp-bots camp-monitor >/dev/null 2>&1 || true", shell=True)
    _ = subprocess.run(["pm2", "start", "arena/day12_pm2/ecosystem.config.js"], check=True)
    time.sleep(2.5)
    print("\n--- 模拟矿工进程意外崩溃 (kill -9) 并观察 PM2 自动重启 ---")
    pid_out = subprocess.check_output(["pm2", "pid", "camp-miner"], text=True).strip()
    if pid_out and pid_out != "0":
        _ = subprocess.run(["kill", "-9", pid_out], check=False)
        time.sleep(2.5)
    _ = subprocess.run(["pm2", "list"], check=True)
    print("\n--- 验证 PM2 自动拉起的修复版矿工正常响应得分 ---")
    _ = subprocess.run([sys.executable, "arena/day11_validator/camp_validator.py", "--score-only", "--seed", "12", "--rounds", "1"], check=True)
else:
    print("提示：当前环境未安装 pm2，可执行 bun install -g pm2（或 npm install -g pm2） 安装后运行上述命令。")

=== arena/day12_pm2/ecosystem.config.js ===
/**
 * 第 12 天配套配置：PM2 进程守护配置 ecosystem.config.js
 * 用途：同时守护共学营矿工（camp-miner）、陪练矿工（camp-bots）与链上监控（camp-monitor）
 *
 * 常用命令（在 camp-kit 根目录执行）：
 * 1. 启动守护：pm2 start arena/day12_pm2/ecosystem.config.js
 * 2. 查看状态：pm2 list   |   查看日志：pm2 logs camp-miner --lines 20
 * 3. 配置日志轮转（单文件最大 50MB，最多保留 10 个历史文件）：
 *    pm2 install pm2-logrotate
 *    pm2 set pm2-logrotate:max_size 50M
 *    pm2 set pm2-logrotate:retain 10
 *    pm2 set pm2-logrotate:compress true
 * 4. 模拟崩溃自愈：kill -9 <camp-miner 的 pid> && sleep 2 && pm2 list
 */

const fs = require("fs");
const path = require("path");

const rootDir = path.resolve(__dirname, "../..");
const venvPython = path.join(rootDir, ".venv", "bin", "python");
const pythonBin = fs.existsSync(venvPython) ? venvPython : "python3";

module.exports = {
  apps: [
    {
      name: "camp-miner",
      cwd: rootDir,
      script: "arena/day09_10_miner/camp_miner.py",
      interpreter: pythonBin,
      args: "--port 8091",
 

[PM2][WARN] Applications camp-miner, camp-bots, camp-monitor not running, starting...
[PM2] App [camp-miner] launched (1 instances)
[PM2] App [camp-bots] launched (1 instances)


[PM2] App [camp-monitor] launched (1 instances)
┌────┬─────────────────┬─────────────┬─────────┬─────────┬──────────┬────────┬──────┬───────────┬──────────┬──────────┬──────────┬──────────┐
│ id │ name            │ namespace   │ version │ mode    │ pid      │ uptime │ ↺    │ status    │ cpu      │ mem      │ user     │ watching │
��────┼─────────────────┼─────────────┼─────────┼─────────┼──────────┼────────┼──────┼───────────┼──────────┼──────────┼──────────┼──────────┤
│ 1  │ camp-bots       │ default     │ N/A     │ fork    │ 16527    │ 0s     │ 0    │ online    │ 0%       │ 14.7mb   │ camp │ disabled │
│ 0  │ camp-miner      │ default     │ N/A     │ fork    │ 16526    │ 0s     │ 0    │ online    │ 0%       │ 14.7mb   │ camp │ disabled │
│ 2  │ camp-monitor    │ default     │ N/A     │ fork    │ 16528    │ 0s     │ 0    │ online    │ 0%       │ 10.6mb   │ camp │ disabled │
└────┴─────────────────┴─────────────┴─────────┴─────────┴──────────┴────────┴──────┴───────────┴──────────┴─��


--- 模拟矿工进程意外崩溃 (kill -9) 并观察 PM2 自动重启 ---


┌────┬─────────────────┬─────────────┬─────────┬─────────┬──────────┬────────┬──────┬───────────┬──────────┬──────────┬──────────┬──────────┐
│ id │ name            │ namespace   │ version │ mode    │ pid      │ uptime │ ↺    │ status    │ cpu      │ mem      │ user     │ watching │
��────┼─────────────────┼─────────────┼─────────┼─────────┼──────────┼────────┼──────┼───────────┼──────────┼──────────┼──────────┼──────────┤
│ 1  │ camp-bots       │ default     │ N/A     │ fork    │ 16527    │ 5s     │ 0    │ online    │ 0%       │ 76.2mb   │ camp │ disabled │
│ 0  │ camp-miner      │ default     │ N/A     │ fork    │ 17007    │ 0s     │ 1    │ online    │ 0%       │ 68.6mb   │ camp │ disabled │
│ 2  │ camp-monitor    │ default     │ N/A     │ fork    │ 16528    │ 5s     │ 0    │ online    │ 0%       │ 73.8mb   │ camp │ disabled │
└────┴─────────────────┴─────────────┴─────────┴─────────┴──────────┴────────┴──────┴───────────┴──────────┴─��────────┴──────────┴──────────┘
host metrics | c

🔑 第 1 步  验证者 UID 1  质押 53.340971543β  验证许可=True  区块 #89
📡 第 2 步  找到 3 个已发布端点的矿工：UID 2@192.168.50.194:8091，UID 3@192.168.50.194:8092，UID 4@192.168.50.194:8093

📝 第 1 轮  task_7252：【教学快讯】Bittensor 生态项目 Beacon 宣布完成 6000万美元 战略轮融资。本轮融资由 Blue Harbor Fund 领投。项目位于 33号 子网，团队表示资金将用于扩充算力节点。
   UID 2   耗时  0.02s  本轮 0.9976  EMA 0.0000→0.2993  通过（字段正确 4/4）
   UID 3   耗时  0.01s  本轮 0.0000  EMA 0.0000→0.0000  依据不在原文中：[项目名称]『AI 总结：数据真实可靠』
   UID 4   耗时  2.02s  本轮 0.6230  EMA 0.0000→0.1869  通过（字段正确 3/4）

📊 第 3 步  归一化权重（只看比例）：UID 2=0.616，UID 3=0.000，UID 4=0.384

（--score-only：本次��提交权重）


> **Day 12 验收清单**：
> 1. 用 `pm2 start arena/day12_pm2/ecosystem.config.js` 同时守护矿工、陪练和监控脚本，截图 `pm2 list` 进程均为 `online`。
> 2. `kill -9` 矿工进程 PID，截图 PM2 自动重启（重启次数 `↺` 加 1），并再跑一次验证者确认矿工恢复服务。
> 3. 配置 `pm2-logrotate`，将单个日志上限设为 `50M`，截图配置结果。

---
## Day 13：用 Docker 封装矿工

当需要在多台云主机上批量部署矿工时，用 Docker 封装能避免 Python 环境冲突。容器化部署有三条关键配置规范：

| 配置项 | 写法与原因 |
| :--- | :--- |
| **热密钥只读挂载（`:ro`）与冷密钥隔离** | 在 `docker-compose.yml` 中配置 `../../.localnet/wallets/miner/hotkeys/default:/root/.bittensor/wallets/miner/hotkeys/default:ro`，只挂载热密钥供节点签名（完全不挂载冷密钥私钥），且用 `:ro` 防止容器内进程篡改宿主机密钥文件 |
| **`.dockerignore` 防止构建泄露** | 构建上下文（`context: ../..`）根目录的 `.dockerignore` 排除了 `.localnet/`、`wallets/`、`*coldkey*`、`.env`，严防 `docker build` 时将本地密钥或环境文件打进镜像层 |
| **容器访问宿主机本地链与 GPU** | 容器内连宿主机本地链需填 `ws://host.docker.internal:9944`（Linux 下配 `extra_hosts: ["host.docker.internal:host-gateway"]`）；若子网需要 GPU，需在 `docker-compose.yml` 中声明 `deploy.resources.reservations.devices`（`driver: nvidia`） |

下面查看根目录 `.dockerignore` 与 `arena/day13_docker/` 下的容器化配置文件，并实测切换为 Docker 容器运行矿工：

In [25]:
# [Day 13.1] 审阅容器安全配置并启动 Docker Compose 容器化矿工
import http.client
import socket
import subprocess
import sys
import time
from pathlib import Path

for cfg_path in [".dockerignore", "arena/day13_docker/Dockerfile", "arena/day13_docker/docker-compose.yml"]:
    print(f"=== {cfg_path} ===")
    print(Path(cfg_path).read_text(encoding="utf-8"))

def wait_port(port: int, timeout: float = 30.0) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("127.0.0.1", port)) == 0:
                return True
        time.sleep(0.5)
    return False

def wait_http_server(port: int, timeout: float = 30.0) -> bool:
    # 等待容器内 Python HTTPServer 真正就绪（避免 macOS Docker 端口代理提前握手导致 RemoteDisconnected）
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            conn = http.client.HTTPConnection("127.0.0.1", port, timeout=2.0)
            conn.request("GET", "/")
            resp = conn.getresponse()
            conn.close()
            if resp.status in (200, 400, 403, 404, 405, 501):
                return True
        except Exception:
            pass
        time.sleep(0.5)
    return False

# 1. 停止宿主机上的 camp-miner 进程，将 8091 端口腾给 Docker 容器（保留宿主机陪练矿工 8092/8093）
subprocess.run("pm2 delete camp-miner camp-bots camp-monitor >/dev/null 2>&1 || true", shell=True)
from arena.common.chain import ensure_port_free
ensure_port_free(8091)
_ = subprocess.run([sys.executable, "arena/day09_10_miner/practice_bots.py", "-d"], check=True)
assert wait_port(8092, 20.0) and wait_port(8093, 20.0), "陪练矿工启动超时"

# 2. 构建并启动 Docker Compose 容器服务
print("⏳ 正在构建并启动 Docker 矿工容器 (camp-miner)...")
_ = subprocess.run(["docker", "compose", "-f", "arena/day13_docker/docker-compose.yml", "build", "-q"], check=True)
_ = subprocess.run(["docker", "compose", "-f", "arena/day13_docker/docker-compose.yml", "up", "-d"], check=True)
assert wait_http_server(8091, 30.0), "Docker 容器矿工 (8091) HTTP 服务启动超时"

_ = subprocess.run(["docker", "compose", "-f", "arena/day13_docker/docker-compose.yml", "ps"], check=True)

print("\n--- 运行验证者确认 Docker 容器内的修复版矿工正常响应拿分 ---")
Path(".localnet/validator_scores.json").unlink(missing_ok=True)
_ = subprocess.run([sys.executable, "arena/day11_validator/camp_validator.py", "--score-only", "--seed", "12", "--rounds", "1"], check=True)

=== .dockerignore ===
# 第 13 天安全红线：严禁将本地私钥、助记词、.env 凭证、本地链钱包或虚拟环境打包进 Docker 镜像
.venv/
.localnet/
__pycache__/
*.pyc
.env
*.pem
*.key
wallets/
.bittensor/
.git/
solutions/

=== arena/day13_docker/Dockerfile ===
# 第 13 天配套文件：Bittensor 矿工与监控服务容器镜像 Dockerfile
FROM python:3.12-slim

WORKDIR /app

# 1. 先复制依赖清单以利用 Docker 缓存层
COPY requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r /app/requirements.txt

# 2. 复制应用代码（注意：.dockerignore 已排除 .env 与所有钱包密钥文件）
COPY tools/ /app/tools/
COPY arena/ /app/arena/

ENV PYTHONUNBUFFERED=1

EXPOSE 8091

# 默认启动共学营矿工服务
CMD ["python", "arena/day09_10_miner/camp_miner.py", "--port", "8091"]

=== arena/day13_docker/docker-compose.yml ===
# 第 13 天配套编排：Docker Compose 部署与密钥只读安全挂载
# 启动命令（在 camp-kit 根目录执行）：
#   docker compose -f arena/day13_docker/docker-compose.yml up -d

services:
  camp-miner:
    build:
      context: ../..
      dockerfile: arena/day13_docker/Dockerfile
    container_name: bittensor-camp-miner
    restart: unless-stopped
    po

ℹ️ 端口 8092 被上次残留的后台进程（PID 17218）占用，已自动停止旧进程并释放端口。
ℹ️ 链上已登记 192.168.50.194:8092（UID 3），无需重复发布。
ℹ️ 链上已登记 192.168.50.194:8093（UID 4），无需重复发布。
🤖 两个陪练矿工已在后台启动（PID 19444，监听 192.168.50.194:8092 / 8093）
📄 实时日志：/workspace/camp-kit/.localnet/bots.log（可执行 tail -f .localnet/bots.log 查看）
⏳ 正在构建并启动 Docker 矿工容器 (camp-miner)...


 Image day13_docker-camp-monitor Building 
 Image day13_docker-camp-miner Building 


 Image day13_docker-camp-miner Built 
 Image day13_docker-camp-monitor Built 
 Container bittensor-camp-miner Creating 
 Container bittensor-camp-monitor Creating 
 Container bittensor-camp-miner Created 
 Container bittensor-camp-monitor Created 
 Container bittensor-camp-monitor Starting 
 Container bittensor-camp-miner Starting 


 Container bittensor-camp-monitor Started 
 Container bittensor-camp-miner Started 


NAME                     IMAGE                       COMMAND                  SERVICE        CREATED        STATUS                  PORTS
bittensor-camp-miner     day13_docker-camp-miner     "python arena/day09_…"   camp-miner     1 second ago   Up Less than a second   0.0.0.0:8091->8091/tcp, [::]:8091->8091/tcp
bittensor-camp-monitor   day13_docker-camp-monitor   "python arena/day11_…"   camp-monitor   1 second ago   Up Less than a second   8091/tcp

--- 运行验证者确认 Docker 容器内的修复版矿工正常响应拿分 ---


🔑 第 1 步  验证者 UID 1  质押 57.440994422β  验证许可=True  区块 #97
📡 第 2 步  找到 3 个已发布端点的矿工：UID 2@192.168.50.194:8091，UID 3@192.168.50.194:8092，UID 4@192.168.50.194:8093

📝 第 1 轮  task_7252：【教学快讯】Bittensor 生态项目 Beacon 宣布完成 6000万美元 战略轮融资。本轮融资由 Blue Harbor Fund 领投。项目位于 33号 子网，团队表示资金将用于扩充算力节点。
   UID 2   耗时  0.04s  本轮 0.9958  EMA 0.0000→0.2987  通过（字段正确 4/4）
   UID 3   耗时  0.02s  本轮 0.0000  EMA 0.0000→0.0000  依据不在原文中：[项目名称]『AI 总结：数据真实可靠』
   UID 4   耗时  2.02s  本轮 0.6235  EMA 0.0000→0.1871  通过（字段正确 3/4）

📊 第 3 步  归一化权重（只看比例）：UID 2=0.615，UID 3=0.000，UID 4=0.385

（--score-only：本次��提交权重）


> **Day 13 验收清单**：
> 1. 先停掉 PM2 里的宿主机矿工（`pm2 delete camp-miner camp-bots camp-monitor`），再用 `docker compose -f arena/day13_docker/docker-compose.yml up -d --build` 启动矿工容器，截图 `docker compose -f arena/day13_docker/docker-compose.yml ps`，并跑一次验证者确认容器内矿工正常得分。
> 2. 截图 `docker-compose.yml` 中的 `:ro` 只读挂载配置，并说明根目录 `.dockerignore` 防止了什么。
> 3. 回答：容器里的矿工要连宿主机上的本地链，链地址应该怎么填？如果换成需要 GPU 的子网，`docker-compose.yml` 要加什么？

---
## Day 14：结营验收

结营验收支持**二选一**提交：
- **方案 A（完整运行记录汇总）**：汇总 Day 8–13 的本地链注册与端点、矿工修复前后得分对比、验证者提交与链上生效权重、以及 PM2/Docker 稳定运行与监控输出。
- **方案 B（学习笔记与避坑手册）**：整理每天至少 3 条核心收获、卡得最久的 3 个问题与解决过程、以及给下一期同学的 5 条建议。

运行下方最后一个单元格，自动从当前本地教学链读取你的矿工与验证者实时账本数据，生成 **方案 A 结营技术验收单**：

In [26]:
# [Day 14] 实时核验本地教学链账本与服务状态，生成结营技术验收单
import bittensor as bt
from arena.common.chain import default_netuid, load_wallet

sub_local = bt.Subtensor("ws://127.0.0.1:9944")
netuid = default_netuid()
mg = sub_local.subnets.metagraph(netuid)

val_hk = load_wallet("validator", "default").hotkey.ss58_address
miner_hk = load_wallet("miner", "default").hotkey.ss58_address
val_uid = sub_local.neurons.uid(val_hk, netuid)
miner_uid = sub_local.neurons.uid(miner_hk, netuid)
assert val_uid is not None and miner_uid is not None, "验证者或矿工未注册"

val_n = mg.neurons[val_uid]
miner_n = mg.neurons[miner_uid]

# 计算你的矿工在子网所有矿工中的真实激励排名
miner_peers = sorted([n for n in mg.neurons if n.uid != val_uid], key=lambda x: float(x.incentive), reverse=True)
my_rank = next(i for i, n in enumerate(miner_peers, 1) if n.uid == miner_uid)

weights_rows = sub_local.weights.weights(netuid)
val_weights = weights_rows.get(val_uid) or {}

print("=" * 74)
print("          Bittensor 14 天开发者共学营 · 链上状态实时核验单")
print("=" * 74)
print(f"1. 链上区块高度          : Block #{mg.block} (Netuid {netuid} 本地教学链)")
print(f"2. 验证者节点 (UID {val_uid})    : Permit={val_n.validator_permit} | Stake={val_n.total_stake} | Dividends={float(val_n.dividends):.4f}")
print(f"3. 链上已生效验证者权重  : { {u: round(w, 4) for u, w in val_weights.items()} }")
print(f"4. 你的矿工   (UID {miner_uid})    : Endpoint={miner_n.axon} | Permit={miner_n.validator_permit}")
print(f"5. 矿工激励占比与真实排名: Incentive = {float(miner_n.incentive):.4f} (矿工排名: 第 {my_rank} / {len(miner_peers)} 名)")
print(f"6. 矿工上个周期排放      : Emission  = {miner_n.emission} / tempo")
print(f"7. 节点当前链上质押      : TotalStake= {miner_n.total_stake} (含累积排放自动质押)")
print("=" * 74)

assert val_n.validator_permit, "验证者尚未持有 validator_permit"
assert bool(miner_n.axon), "矿工尚未发布链上端点"
assert my_rank == 1 and float(miner_n.incentive) > 0.5, f"矿工尚未取得第 1 名激励 (当前排名 {my_rank}, Incentive={miner_n.incentive})"
print("🎉 全链路闭环（钱包 -> 元图 -> 矿工排障 -> 验证者打分上链 -> 排放到账 -> 容器守护）实测核验通过！")

          Bittensor 14 天开发者共学营 · 链上状态实时核验单
1. 链上区块高度          : Block #97 (Netuid 2 本地教学链)
2. 验证者节点 (UID 1)    : Permit=True | Stake=57.440994422β | Dividends=1.0000
3. 链上已生效验证者权重  : {2: 0.5952, 4: 0.4048}
4. 你的矿工   (UID 2)    : Endpoint=192.168.50.194:8091 | Permit=False
5. 矿工激励占比与真实排名: Incentive = 0.5952 (矿工排名: 第 1 / 4 名)
6. 矿工上个周期排放      : Emission  = 2.440330221β / tempo
7. 节点当前链上质押      : TotalStake= 4.880660445β (含累积排放自动质押)
🎉 全链路闭环（钱包 -> 元图 -> 矿工排障 -> 验证者打分上链 -> 排放到账 -> 容器守护）实测核验通过！
